In [1]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import time
import numpy as np
from bs4 import BeautifulSoup as bs

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error

from geopy.geocoders import Nominatim
from geopy.distance import geodesic

In [2]:
def juntar(lista):
    merged = defaultdict(list)
    for d in lista:
        for k, v in d.items():
            merged[k].append(v)
    return merged

In [3]:
resultdf = pd.read_csv("todo3.csv", encoding='latin1')
def fix_mojibake(x):
    if isinstance(x, str):
        try:
            return x.encode("latin1").decode("utf-8")
        except:
            return x
    return x
for col in resultdf.select_dtypes(include="object").columns:
    resultdf[col] = resultdf[col].apply(fix_mojibake)

FileNotFoundError: [Errno 2] No such file or directory: 'todo3.csv'

In [ ]:
cambiarfase={'Semi-finals':'Semifinals','QUARTERFINALS':'Quarterfinals','Semifinal':'Semifinals'
            ,'-Quarterfinals':'Quarterfinals', 'Quartefinals':'Quarterfinals', 'Quarterfinal':'Quarterfinals',
            'SEMIFINALS':'Semifinals', 'FINAL': 'Final', 'Finals': 'Final','3rd-Place':'Place',
            'Quarter-Finals':'Quarterfinals','final':'Final','Quarter-finals':'Quarterfinals'}

In [ ]:
def cambFase(x):
    try:
        y=int(x.split()[-1])
        if y > 2000:
            long = len(x.split())
            if long==1:
                return x.split()[-1]
            else:
                return x.split()[-2]
        else:
            return x.split()[-1]
    except ValueError:
        return x.split()[-1]

In [ ]:
resultdf=resultdf[(resultdf["homeTeamId"].notna())]
resultdf=resultdf[(resultdf["stage"].notna())]
resultdf["phase"]=resultdf["stage"].apply(lambda x: cambFase(x))
resultdf["phase"]=resultdf["phase"].replace(cambiarfase)
resultdf["leaguePhase"]=resultdf["leagueName"] + resultdf["phase"]

In [ ]:
internacionalesAbr = ["Men's International Friendly", "Africa Cup of Nations", "UEFA Nations League", 
                   "FIFA World Cup", "International Friendly", "Copa América", "Concacaf Nations League"
                   , "Gold Cup", "Women's International Friendly", "FIFA U-17 World Cup", "ASEAN Champ"
                   , "WCQ - Concacaf", "Confederations Cup", "AFC Asian Cup", "Men's Olympic Soccer Tournament"
                   , "AFC ASIAN CUP - GROUP F", "Concacaf W Championship", "WCQ - AFC/Concacaf Playoff"
                   , "Women's Olympic Soccer Tournament", "Women's EURO", "AFC Cup Qualifying", "Tournoi de France"
                   , "FIFA Women's World Cup", "Friendly", "WCQ - CAF", "EURO", "WCQ - CONMEBOL", "COSAFA Cup"
                   , "WCQ - AFC", "CHAN Qualifying", "EURO Under-19", "EURO Qualifying", "WCQ - UEFA"
                   , "Women's Nations League", "EURO Under-21", "Gold Cup Qualifying", "WCQ - CONMEBOL/OFC","Concacaf Women's Olympic Qualifying"
                   , "WCQ - Concacaf/OFC Playoff", "AFC/CONCACAF Playoffs", "Arabian Gulf Cup", "AFCON Qualifying"
                   , "CHAN", "Concacaf Nations League Qualifying","Copa América Femenina","FIFA U-17 Women's World Cup"
                   , "FIFA U-20 World Cup", "Pinatar Cup", "SAFF Championship", "SheBelieves Cup", "WAFU Cup of Nations"
                   , "W Gold Cup", "WCQ", "WCQ - CONMEBOL/OFC", "WCQ - OFC", "WWCQ - UEFA", "WWCQ - Playoff Tournament"
                   , "AFC/CONCACAF Playoffs", "2026 FIFA World Cup¤ AFC Qualify","Eliminatórias - Copa América"
                   , "EURO U-21 Qualifying", "Sudamericano Sub-20", "U20 Intercontinental Cup", "Panamericanos – Hombres"
                   , "Panamericanos – Mujeres", "2021 U21 European Championship -", "2017 U21 EUROPEAN CHAMPIONSHIP -"
                   , "Nations", "Concacaf Women's Olympic Qualifying", "2021 U21 Int. Friendly","TOULON TOURNAMENT -  GROUP A"
                   , "Men's U-21 Friendly", "Arnold Clark Cup", "Algarve Cup", "Conf Pl","Algrave Cup", "Torneio Futebol Feminino"
                   , "TOULON TOURNAMENT -  GROUP C", "TOULON TOURNAMENT -  GROUP B"]

In [ ]:
paises = {
    "Cape Verde Islands": "Cape Verde",
    "North Koreo": "North Korea",
    "Curaçao" : "Curacao",
    "Ireland":"Republic of Ireland",
    "Korea Republic": "South Korea",
    "Congo" : "Congo DR",
    "China PR": "China",
    "Turkey": "Türkiye",
    "Russia ": "Russia",
    "USA":"United States",
    "IR Iran": "Iran"
}
resultdf["country"] = resultdf["country"].replace(paises)
resultdf["homeTeamName"] = resultdf["homeTeamName"].replace(paises)
resultdf["awayTeamName"] = resultdf["awayTeamName"].replace(paises)

In [ ]:
df_pai2 = resultdf[resultdf["leagueAbbr"].isin(internacionalesAbr)]
df_clubs2=resultdf[~resultdf["leagueAbbr"].isin(internacionalesAbr)]

comb2 = np.concatenate((df_pai2["homeTeamName"],df_pai2["awayTeamName"]))
uniq2 = np.unique(comb2)
mask = ~np.isin(uniq2,['Flamengo U20','Juventus'])
uniq2 = uniq2[mask]
a_quit2=df_clubs2[(df_clubs2["homeTeamName"].isin(uniq2)) | (df_clubs2["awayTeamName"].isin(uniq2))]
df_pai2 = pd.concat([df_pai2,a_quit2])
df_clubs2 = df_clubs2.drop(a_quit2.index, axis=0)

resultdf["is_national"] = resultdf["gameId"].isin(df_pai2["gameId"])

In [ ]:
ligas=resultdf["leagueName"].unique()
confs = ["UEFA", "AFC", "CONCACAF", "CONMEBOL", "CAF", "OFC", "FIFA", "Club"
          "Intercontinental","World","Nations","Friendly", "Olympic", 
         "Olympics", "International", "Semifinals", "Finals", "Quarterfinals", 
         "Round", "Sudamericana"]
ligas_ints=[]
for liga in ligas:
    if pd.isna(liga):
            continue 
    if any(i.lower() in liga.lower() for i in confs):
        ligas_ints.append(liga)

In [ ]:
#resultdf.groupby("homeTeamName").count().sort_values(by="homeTeamId").tail(40)
santos = {225:"Santos Laguna",913:"Santos de Guapiles",7070:"Santos ZA"}
liverpool = {5492:"Liverpool F.C."}
olimpia = {884:"C.D. Olimpia"}
river = {5498:"River Plate UY"}
nacional = {5584:"Club Nacional"}
alianza = {7225:"Alianza FC San Salvador"}
coms = {14074:"Club Comunicaciones"}
atlas = {10099:"Club Atlético Atlas"}
gua = {3448:"Guarani FC"}

trad = {"Santos":santos,"Liverpool":liverpool,"Olimpia":olimpia,"River Plate":river,
       "Nacional":nacional,"Alianza FC":alianza,"Comunicaciones":coms,"Atlas":atlas,
       "Guarani":gua}

for base_name, team_dict in trad.items():
    for team_id, correct_name in team_dict.items():
        df = (((resultdf["awayTeamId"]==team_id) | (resultdf["homeTeamId"]==team_id)))
        resultdf.loc[df,"homeTeamName"]=resultdf[df]["homeTeamName"].replace({base_name:correct_name})
        resultdf.loc[df,"awayTeamName"]=resultdf[df]["awayTeamName"].replace({base_name:correct_name})


In [ ]:
map_ciudades={"Torreón":"Torreon", "King Fahd International Stadium": "Riyadh",
              "Ciudad Autónoma de Buenos Aires":"Buenos Aires",
              "Los Angeles, California": "Los Angeles",
              "León de los Aldamas":"Leon",
    "Los Angles, California": "Los Angeles","Inglewood, California": "Inglewood","Irvine, California":"Irvine",
    "Pasadena, California": "Pasadena","Carson, California": "Carson","Long Beach, California": "Long Beach",
    "Charlotte, North Carolina": "Charlotte","Matthews, North Carolina": "Charlotte",

    "New York, New York": "New York City","New York City, New York": "New York City",
    "Brooklyn": "New York City","The Bronx, New York": "New York City","Brooklyn, New York": "New York City",
    "East Rutherford, New Jersey": "East Rutherford","Harrison, New Jersey": "New York City",
              
    "Washington": "Washington, D.C.","Cincinnati, Ohio": "Cincinnati",
    "Washington, District of Columbia": "Washington, D.C.",

    "Miami, Florida": "Miami","Miami Gardens, Florida": "Miami Gardens",
    "Fort Lauderdale, Florida": "Fort Lauderdale",

    "Boston, Massachusetts": "Boston","Foxborough, Massachusetts": "Foxborough",

    "Chicago, Illinois": "Chicago","Bridgeview, Illinois": "Bridgeview",

    "Houston, Texas": "Houston",

    "Dallas, Texas": "Dallas","Arlington, Texas": "Arlington","Toyota Stadium":"Dallas",

    "San Jose, California": "San Jose","Santa Clara, California": "Santa Clara",
    "Stanford, California": "Stanford",

    "San Diego, California": "San Diego",

    "Seattle, Washington": "Seattle","Tacoma, Washington": "Tacoma","Tukwila, Washington": "Tukwila",

    "Portland, Oregon": "Portland",

    "Kansas City, Missouri": "Kansas City","Kansas City, Kansas": "Kansas City",

    "Nashville, Tennessee": "Nashville",

    "Salt Lake City, Utah": "Salt Lake City","Sandy, Utah": "Salt Lake City","Herriman, Utah": "Herriman",

    "San Francisco, California": "San Francisco","Oakland, California": "Oakland",

    "Phoenix, Arizona": "Phoenix","Scottsdale, Arizona": "Scottsdale","Chandler, Arizona": "Chandler",

    "Orlando, Florida": "Orlando","Kissimmee": "Kissimmee",

    "Tampa, Florida": "Tampa","St. Petersburg, Florida": "Tampa",

    "Detroit, Michigan": "Detroit","Hamtramck, Michigan": "Hamtramck",

    "Cleveland, Ohio": "Cleveland","Columbus, Ohio": "Columbus",

    "Minneapolis, Minnesota": "Minneapolis","Saint Paul, Minnesota": "Saint Paul",
    
    "Atlanta, Georgia": "Atlanta","North West Atlanta, Georia":"Atlanta",

    "Chester, Pennsylvania": "Chester","Wayne, Pennsylvania": "Wayne","Philadelphia, Pennsylvania": "Philadelphia"}

resultdf["city"]=resultdf["city"].replace(map_ciudades)

In [ ]:
resultdf["is_international"] = ((resultdf["is_national"]==1) 
                            | (resultdf["leagueName"].isin(ligas_ints)))

In [ ]:
paisss={}
for i in resultdf["country"].unique():
    if i in uniq2:
        paisss[i]=i


paisss["St. Lucia"]="St. Lucia"
paisss["Aruba"]="Aruba"
paisss["Sint Maarten"]="Sint Maarten"

In [ ]:
team_venue = (
    resultdf[resultdf["is_national"]==0]
    .dropna(subset=["venueId"])
    .groupby("homeTeamId")["venueId"]
    .agg(lambda x: x.value_counts().idxmax())
)

venue_name = (
    resultdf
    .dropna(subset=["fullName"])
    .groupby("venueId")["fullName"]
    .agg(lambda x: x.value_counts().idxmax())
)

team_id = (
    resultdf
    .dropna(subset=["homeTeamId"])
    .groupby("homeTeamName")["homeTeamId"]
    .agg(lambda x: x.value_counts().idxmax())
)

id_name = (
    resultdf
    .dropna(subset=["homeTeamName"])
    .groupby("homeTeamId")["homeTeamName"]
    .agg(lambda x: x.value_counts().idxmax())
)

tabla_city = (
    resultdf
    .dropna(subset=["city"])
    .groupby("venueId")["city"]
    .agg(lambda x: x.value_counts().idxmax())
)

tabla_country = (
    resultdf
    .dropna(subset=["country"])
    .groupby("venueId")["country"]
    .agg(lambda x: x.value_counts().idxmax())
)
ordinal_round_pattern = r"^\s*\d+(st|nd|rd|th)\s+round\s*$"

invalid_league_pattern = (
    r"^\s*("
    r"group|grupo|"
    r"zone|zona|"
    r"round|ronda|"
    r"temporada|torneo|"
    r")\s+[a-z0-9ivx]+"
    r"\s*$"
)

league_country = (
    resultdf[
        (resultdf["is_international"] == 0)
        & resultdf["country"].notna()
        & ~resultdf["leagueName"].astype(str).str.match(
            invalid_league_pattern, case=False
        )
        & ~resultdf["leagueName"].astype(str).str.match(
            ordinal_round_pattern, case=False
        )
    ]
    .groupby("leagueName")["country"]
    .agg(lambda x: x.value_counts().idxmax())
    #.to_dict()
)

team_country = (
    resultdf[resultdf["is_national"]==0]
    .dropna(subset=["country"])
    .groupby(["homeTeamId"])["country"]
    .agg(lambda x: x.value_counts().idxmax())
)

team_city = (
    resultdf[resultdf["is_national"]==0]
    .dropna(subset=["city"])
    .groupby(["homeTeamId"])["city"]
    .agg(lambda x: x.value_counts().idxmax())
)

city_country = (
    resultdf
    .dropna(subset=["country"])
    .groupby(["city"])["country"]
    .agg(lambda x: x.value_counts().idxmax())
)

team_venue[220]=6351 
team_city[220] = "Guadalupe"
team_city[7279] = "Auckland"
team_country[7279] = "New Zealand"
city_country["Whangarei"] = "New Zealand"
team_id["United States"]=660

In [ ]:
wc2026=(((resultdf["country"]=="Canada") | (resultdf["country"]=="Mexico") 
                   | (resultdf["country"]=="United States")) & (resultdf["leagueName"]=="FIFA World Cup")
                 & (resultdf["gameId"]>=750486))

ganadores = {
        "Winner Playoff Path D": "Czechia",
    "Winner Playoff Path A": "Bosnia and Herzegovina",
    "Winner Playoff Path C": "Türkiye",
    "Winner Playoff Path B": "Sweden",
    "Intercontinental Playoff Path 2": "Iraq",
    "Intercontinental Playoff Path 1": "Congo DR",
    "2A": "Czechia","1A": "Mexico",
    "2B": "Bosnia and Herzegovina","1B": "Switzerland",
    "2C": "Brazil","1C": "Morocco",
    "2D": "Türkiye","1D": "United States",
    "2E": "Ivory Coast","1E": "Germany",
    "2F": "Japan","1F": "Netherlands",
    "2G": "Egypt","1G": "Belgium",
    "2H": "Uruguay","1H": "Spain",
    "Group I 2nd Place": "Senegal","Group I Winner": "France",
    "Group J 2nd Place": "Algeria","Group J Winner": "Argentina",
    "Group K 2nd Place": "Colombia","Group K Winner": "Portugal",
    "Group L 2nd Place": "England","Group L Winner": "Croatia",
    "Third Place Group A/B/C/D/F":"Scotland","Third Place Group C/D/F/G/H":"Sweden","Third Place Group C/E/F/H/I":"Saudi Arabia",
    "Third Place Group E/H/I/J/K":"Norway","Third Place Group A/E/H/I/J":"South Africa","Third Place Group B/E/F/I/J":"Qatar",
    "Third Place Group E/F/G/I/J":"Ecuador","Third Place Group D/E/I/J/L":"Australia",
    "Round of 32 1 Winner":"Czechia","Round of 32 2 Winner":"Germany","Round of 32 3 Winner":"Brazil",
    "Round of 32 4 Winner":"Morocco","Round of 32 5 Winner":"France","Round of 32 6 Winner":"Senegal",
    "Round of 32 7 Winner":"Mexico","Round of 32 8 Winner":"Croatia","Round of 32 9 Winner":"United States",
    "Round of 32 10 Winner":"Belgium","Round of 32 11 Winner":"England","Round of 32 12 Winner":"Spain",
    "Round of 32 13 Winner":"Ecuador","Round of 32 14 Winner":"Argentina","Round of 32 15 Winner":"Portugal",
    "Round of 32 16 Winner":"Türkiye","Round of 16 1 Winner":"Germany","Round of 16 2 Winner":"Brazil",
    "Round of 16 3 Winner":"Senegal","Round of 16 4 Winner":"Croatia","Round of 16 5 Winner":"England",
    "Round of 16 6 Winner":"Belgium","Round of 16 7 Winner":"Argentina","Round of 16 8 Winner":"Portugal",
    "Quarterfinal 1 Winner":"Germany","Quarterfinal 2 Winner":"England","Quarterfinal 3 Winner":"Croatia",
    "Quarterfinal 4 Winner":"Portugal","Semifinal 1 Winner":"Germany","Semifinal 2 Winner":"Portugal",
    "Semifinal 1 Loser":"Croatia","Semifinal 2 Loser":"England"
}
resultdf.loc[wc2026, "homeTeamName"] = resultdf["homeTeamName"].replace(ganadores)
resultdf.loc[wc2026, "awayTeamName"] = resultdf["awayTeamName"].replace(ganadores)
resultdf.loc[wc2026, "homeTeamId"] = resultdf["homeTeamName"].map(team_id)
resultdf.loc[wc2026, "awayTeamId"] = resultdf["awayTeamName"].map(team_id)
resultdf.loc[wc2026, "venueId"] = resultdf["venueId"].replace({10660:3879,10897:1413})
resultdf.loc[wc2026, "phase"] = resultdf["phase"].replace({"32":"16","Match":"Place"})
resultdf.loc[wc2026, "leaguePhase"] = resultdf["leaguePhase"].replace({"FIFA World Cup32":"FIFA World Cup16",
                                                                      "FIFA World CupMatch":"FIFA World CupPlace"})

In [ ]:
mask_club = resultdf["is_national"] == 0

resultdf.loc[mask_club, "home_country"] = (
    resultdf.loc[mask_club, "homeTeamId"].map(team_country)
)

resultdf.loc[mask_club, "away_country"] = (
    resultdf.loc[mask_club, "awayTeamId"].map(team_country)
)

mask_nat = resultdf["is_national"] == 1

resultdf.loc[mask_nat, "home_country"] = (
    resultdf.loc[mask_nat, "homeTeamName"].map(paisss)
)

resultdf.loc[mask_nat, "away_country"] = (
    resultdf.loc[mask_nat, "awayTeamName"].map(paisss)
)

resultdf["home_city"] = resultdf["homeTeamId"].map(team_city)
resultdf["away_city"] = resultdf["awayTeamId"].map(team_city)

resultdf["home_local"] = (
    (
        (resultdf["is_national"] == 1) &
        (resultdf["home_country"] == resultdf["country"])
    ) |
    (
        (resultdf["is_national"] == 0) &
        (resultdf["home_city"] == resultdf["city"])
    )
).astype(int)

resultdf["away_local"] = (
    (
        (resultdf["is_national"] == 1) &
        (resultdf["away_country"] == resultdf["country"])
    ) |
    (
        (resultdf["is_national"] == 0) &
        (resultdf["away_city"] == resultdf["city"])
    )
).astype(int)

In [ ]:
resultdf["is_international"] = ((resultdf["is_national"]==1) 
                            | (resultdf["leagueName"].isin(ligas_ints)) 
                            | ((resultdf["home_country"]!=resultdf["country"]) | (resultdf["away_country"]!=resultdf["country"])))

In [ ]:
mask = (
    resultdf["country"].isna()
    & resultdf["city"].notna()
    & resultdf["leagueName"].map(league_country).notna()
    & (resultdf["city"].map(city_country) == resultdf["leagueName"].map(league_country))
)

resultdf.loc[mask, "country"] = resultdf.loc[mask, "city"].map(city_country)
#resultdf[mask]

In [ ]:
mask=(resultdf["country"]=="South Africa")

resultdf.loc[mask, ["city", "country"]] = pd.NA

resultdf.loc[mask, "city"] = (
    resultdf.loc[mask, "venueId"].map(tabla_city)
)

resultdf.loc[mask, "country"] = (
    resultdf.loc[mask, "venueId"].map(tabla_country)
)

In [ ]:
resultdf.loc[(resultdf["venueId"]==0), ["venueId"]] = pd.NA

In [ ]:
mask = (resultdf["is_international"] == 0)

resultdf.loc[mask, "venueId"] = (
    resultdf.loc[mask, "venueId"]
    .fillna(resultdf.loc[mask, "homeTeamId"].map(team_venue))
)

#resultdf["venueId"] = resultdf["venueId"].fillna(resultdf["homeTeamName"].map(team_venue))
resultdf["fullName"] = resultdf["fullName"].fillna(resultdf["venueId"].map(venue_name))
resultdf["city"] = resultdf["city"].fillna(resultdf["venueId"].map(tabla_city))
resultdf["country"] = resultdf["country"].fillna(resultdf["venueId"].map(tabla_country))
resultdf["country"] = resultdf["country"].fillna(resultdf["leagueName"].map(league_country))
#resultdf["country"] = resultdf["country"].fillna(resultdf["homeTeamName"].map(team_country))
#resultdf["country"] = resultdf["country"].fillna(resultdf["homeTeamName"].map(paisss))
#resultdf[mask,"city"] = resultdf["city"].fillna(resultdf["homeTeamName"].map(team_city))

resultdf.loc[mask, "country"] = (
    resultdf.loc[mask, "country"]
    .fillna(resultdf.loc[mask, "homeTeamId"].map(team_country))
)

resultdf.loc[mask, "country"] = (
    resultdf.loc[mask, "country"]
    .fillna(resultdf.loc[mask, "homeTeamName"].map(paisss))
)

resultdf.loc[mask, "city"] = (
    resultdf.loc[mask, "city"]
    .fillna(resultdf.loc[mask, "homeTeamId"].map(team_city))
)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
resultdf["fecha"]=pd.to_datetime(resultdf["fecha"])
resultdf["day"]=resultdf["fecha"].dt.day
resultdf["month"]=resultdf["fecha"].dt.month
resultdf["year"]=resultdf["fecha"].dt.year
resultdf["hour"]=resultdf["fecha"].dt.hour

resultdf["day_week"] = pd.to_datetime(resultdf[["year", "month", "day"]]).dt.day_name()

In [ ]:
sinna = resultdf[(resultdf["attendance"].notna()) & (resultdf["attendance"] != 0) & (resultdf["away_local"].notna())
& ((resultdf["country"].notna()) & (resultdf["city"].notna()) & (resultdf["fullName"].notna()) & (resultdf["home_local"].notna())
& (resultdf["venueId"] != 0))].sort_values(by="attendance")

paises = sinna["country"].unique()

for i in paises:
    if pd.isna(i):
        continue

    cities = sinna.loc[sinna["country"] == i, "city"].unique()

    for j in cities:
        if pd.isna(j):
            continue

        mapped_country = city_country.get(j)

        if mapped_country is None or pd.isna(mapped_country):
            continue

        if mapped_country != i:
            print(f"{j} is not in {i} but in {mapped_country}")

sinna["home_country"] = sinna["homeTeamName"].map(team_country)
sinna["away_country"] = sinna["awayTeamName"].map(team_country)
sinna["home_country"] = sinna["homeTeamName"].map(paisss)
sinna["away_country"] = sinna["awayTeamName"].map(paisss)

sinna["home_city"] = sinna["homeTeamName"].map(team_city)
sinna["away_city"] = sinna["awayTeamName"].map(team_city)

#sinna["home_local"] = ((sinna["is_national"]==1) & (sinna["home_country"] == sinna["country"])).astype(int)
#sinna["away_local"] = ((sinna["is_national"]==1) & (sinna["away_country"] == sinna["country"])).astype(int)

#sinna["home_local"] = ((sinna["is_national"]==0) & (sinna["home_city"] == sinna["city"])).astype(int)
#sinna["away_local"] = ((sinna["is_national"]==0) & (sinna["away_city"] == sinna["city"])).astype(int)

sinna["home_local"] = (
    (
        (sinna["is_national"] == 1) &
        (sinna["home_country"] == sinna["country"])
    ) |
    (
        (sinna["is_national"] == 0) &
        (sinna["home_city"] == sinna["city"])
    )
).astype(int)

sinna["away_local"] = (
    (
        (sinna["is_national"] == 1) &
        (sinna["away_country"] == sinna["country"])
    ) |
    (
        (sinna["is_national"] == 0) &
        (sinna["away_city"] == sinna["city"])
    )
).astype(int)

sinna["is_international"] = ((sinna["is_national"]==1) 
                            | (sinna["leagueName"].isin(ligas_ints)) 
                            | ((sinna["home_country"]!=sinna["country"]) | (sinna["away_country"]!=sinna["country"])))

In [ ]:
sinna["neutral_venue"] = (
    (sinna["home_local"] == 0) &
    (sinna["away_local"] == 0)
).astype(int)

In [ ]:
sinna["attendance"] = sinna["attendance"].astype(float)

city_pop= (sinna.groupby("city")["attendance"].mean())
stadium_pop=(sinna.groupby("venueId")["attendance"].mean())
country_pop=(sinna.groupby("country")["attendance"].mean())
league_phase=(sinna.groupby("leaguePhase")["attendance"].mean())
phase_pop=(sinna.groupby("phase")["attendance"].mean())
league_pop=(sinna.groupby("leagueName")["attendance"].mean())
stad_max=(sinna.groupby("venueId")["attendance"].max())

month_pop = (sinna.groupby("month")["attendance"].mean())
hour_pop = (sinna.groupby("hour")["attendance"].mean())
day_pop = (sinna.groupby("day_week")["attendance"].mean())

#home_pop=(sinna[sinna["home_local"] == 1].groupby("homeTeamId")["attendance"].mean())
#away_pop= (sinna[sinna["away_local"]==0].groupby("awayTeamId")["attendance"].mean())

home_local = sinna[sinna["home_local"] == 1][["homeTeamId","attendance"]]
away_local = sinna[sinna["away_local"] == 1][["awayTeamId","attendance"]]
home_local2 = sinna[sinna["home_local"] == 0][["homeTeamId","attendance"]]
away_local2 = sinna[sinna["away_local"] == 0][["awayTeamId","attendance"]]

home_local = home_local.rename(columns={"homeTeamId":"team"})
away_local = away_local.rename(columns={"awayTeamId":"team"})
home_local2 = home_local2.rename(columns={"homeTeamId":"team"})
away_local2 = away_local2.rename(columns={"awayTeamId":"team"})

combined = pd.concat([home_local, away_local])
combined2 = pd.concat([home_local2, away_local2])

home_pop = combined.groupby("team")["attendance"].mean()
away_pop = combined2.groupby("team")["attendance"].mean()





sinna["home_popularity"] = np.where(sinna["home_local"] == 1,
    sinna["homeTeamId"].map(home_pop),
    sinna["homeTeamId"].map(away_pop))
sinna["away_popularity"] = np.where(sinna["away_local"] == 1,
    sinna["awayTeamId"].map(home_pop),
    sinna["awayTeamId"].map(away_pop))

sinna["league_popularity"] = sinna["leagueName"].map(league_pop)
sinna["city_popularity"] = sinna["city"].map(city_pop)
sinna["stadium_popularity"] = sinna["venueId"].map(stadium_pop)
sinna["country_popularity"] = sinna["country"].map(country_pop)
sinna["phase_popularity"] = sinna["phase"].map(phase_pop)#sinna["leaguePhase"].map(league_phase)
sinna["leaguephase_popularity"] = sinna["leaguePhase"].map(league_phase)
sinna["stadium_max"] = sinna["venueId"].map(stad_max)

sinna["day_pop"] = sinna["day_week"].map(day_pop)
sinna["hour_pop"] = sinna["hour"].map(hour_pop)
sinna["month_pop"] = sinna["month"].map(month_pop)

In [ ]:
def dist_ciudades(cid1,cid2):
    geolocator = Nominatim(user_agent="city_distance_calc")
    
    loc1 = geolocator.geocode(cid2)
    loc2 = geolocator.geocode(cid1)
    if loc1 and loc2:
        coords_1 = (loc1.latitude, loc1.longitude)
        coords_2 = (loc2.latitude, loc2.longitude)
    
        distance_km = geodesic(coords_1, coords_2).km
        return distance_km
    else:
        return None

mask= (sinna["is_national"] == 0)
sinna.loc[mask,"home_country"] = sinna["homeTeamId"].map(team_country)
sinna.loc[mask,"away_country"] = sinna["awayTeamId"].map(team_country)


sinna.loc[mask,"home_city"] = sinna["homeTeamId"].map(team_city)
sinna.loc[mask,"away_city"] = sinna["awayTeamId"].map(team_city)

mask = (sinna["is_national"] == 1)
sinna.loc[mask,"home_country"] = sinna["homeTeamName"].map(paisss)
sinna.loc[mask,"away_country"] = sinna["awayTeamName"].map(paisss)


sinna=sinna[(sinna["home_country"].notna()) | (sinna["away_country"].notna())]

#ggg=sinna[(sinna["away_city,country"].isna()) & (sinna["is_national"]==0)]["gameId"]
#sinna=sinna[~sinna["gameId"].isin(ggg)]


mask= (sinna["is_national"] == 0)
sinna.loc[mask]=sinna[(sinna["home_city"].notna()) | (sinna["away_city"].notna())]
sinna.loc[:, "city,country"] = sinna["city"] + ", " + sinna["country"]
sinna.loc[mask, "home_city,country"] = sinna["home_city"] + ", " + sinna["home_country"]
sinna.loc[mask, "away_city,country"] = sinna["away_city"] + ", " + sinna["away_country"]

In [ ]:
cwc=sinna[((sinna["country"]=="United States") & (sinna["leagueName"]=="FIFA Club World Cup"))]
wc2006=sinna[((sinna["year"]==2006) & (sinna["leagueName"]=="FIFA World Cup"))]
wc2010=sinna[((sinna["country"]=="South Africa") & (sinna["leagueName"]=="FIFA World Cup"))]
wc2014=sinna[((sinna["country"]=="Brazil") & (sinna["leagueName"]=="FIFA World Cup"))]
wc2018=sinna[((sinna["country"]=="Russia") & (sinna["leagueName"]=="FIFA World Cup"))]
wc2022=sinna[((sinna["country"]=="Qatar") & (sinna["leagueName"]=="FIFA World Cup"))]

sinna.loc[mask, "distance_home"] = sinna[mask].apply(
    lambda x: dist_ciudades(x["city,country"], x["home_city,country"])
    if pd.notna(x["city,country"]) and pd.notna(x["home_city,country"])
    else None,
    axis=1
)
sinna.loc[mask, "distance_away"] = sinna[mask].apply(
    lambda x: dist_ciudades(x["city,country"], x["away_city,country"])
    if pd.notna(x["city,country"]) and pd.notna(x["away_city,country"])
    else None,
    axis=1
)


mask= (sinna["is_national"] == 1)
sinna.loc[mask, "distance_home"] = sinna[mask].apply(
    lambda x: dist_ciudades(x["city,country"], x["home_country"])
    if pd.notna(x["city,country"]) and pd.notna(x["home_country"])
    else None,
    axis=1
)
sinna.loc[mask, "distance_away"] = sinna[mask].apply(
    lambda x: dist_ciudades(x["city,country"], x["away_country"])
    if pd.notna(x["city,country"]) and pd.notna(x["away_country"])
    else None,
    axis=1
)

In [ ]:
features = [
    "home_popularity",
    "away_popularity",
    "city_popularity",
    "country_popularity",
    "league_popularity",
    "stadium_popularity",
    "phase_popularity",
    "leaguephase_popularity",
    "stadium_max",
    "hour_pop",
    "month_pop",
    "day_pop",
    #leagueName",
    #"country",
    #"city",
    #"is_national",
    #"home_local",
    #"away_local",
    #"neutral_venue"
]
df_pai = sinna[sinna["is_national"]==1]
df_club = sinna[sinna["is_national"]==0]
df_model = sinna[features + ["attendance"]].dropna()
#df_model1 = df_club[features + ["attendance"]].dropna()
#df_model = df_model[df_model["attendance"] > 1000]

In [ ]:
acamb = {4087:4905, 6971: 8461, 4643:4904,4418:6351, 7485:1264, 1421: 4907,
         4485:10646, 2242:1252, 8993:4913, 8674:4910,7604:4908,4727: 1672,7605.0:7605.0}
cwcmx = cwc.drop(columns=["fullName","city","country"])
cwcmx["venueId"]=cwcmx["venueId"].replace(acamb)
cwcmx["fullName"] = cwcmx["venueId"].map(venue_name)
cwcmx["city"] = cwcmx["venueId"].map(tabla_city)
cwcmx["country"] = cwcmx["venueId"].map(tabla_country)

In [ ]:
X = df_model[features]
y = df_model["attendance"]

#cat = ["country","city"]#,"leagueName"]
num = [
    "home_popularity",
    "away_popularity",
    "city_popularity",
    "country_popularity",
    "league_popularity",
    "stadium_popularity",
    "phase_popularity",
    "leaguephase_popularity",
    "stadium_max",
    "hour_pop",
    "month_pop",
    "day_pop",
    #"team_pop_gap",
    #"stadium_league_pop",
    #"is_national",
    #"home_local",
    #"away_local",
    #"neutral_venue"
]

preprocess = ColumnTransformer([
  #  ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ("num", "passthrough", num)
])

#model = RandomForestRegressor( n_estimators=300, max_depth=12, 
#                              min_samples_leaf=8, #15
#                              n_jobs=-1, random_state=42 )

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=14,
    min_samples_leaf=8,   # CLAVE
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)


pipe = Pipeline([
    ("prep", preprocess),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe.fit(X_train, y_train)

In [ ]:
pred = pipe.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred))

In [ ]:
def predecir_asistencia(home, away, country,city,league,venueId, is_national, phase,day,month,hour):
    # flags de localía
    if is_national:
        home_country = id_name.get(home)
        away_country = id_name.get(away)

        home_local = int(home_country == country)
        away_local = int(away_country == country)
    else:
        home_city = team_city.get(home)
        away_city = team_city.get(away)
        
        home_local = int(home_city == city)
        away_local = int(away_city == city)
        
    if ((home_local ==0) & (away_local ==0)):
        neutral_venue=1
    else:
        neutral_venue = 0
        
    
    if home_local:
        home_popularity = home_pop.get(home, home_pop.mean())
    else:
        home_popularity = away_pop.get(home, away_pop.mean())

    if away_local:
        away_popularity = home_pop.get(away, home_pop.mean())
    else:
        away_popularity = away_pop.get(away, away_pop.mean())
    
    city_popularity = city_pop.get(city, city_pop.max())
    country_popularity = country_pop.get(country, country_pop.max())
    league_popularity = league_pop.get(league, league_pop.max())
    stadium_popularity = stadium_pop.get(venueId, stadium_pop.max())
    leaguep = league + phase
    phase_popularity = phase_pop.get(phase, phase_pop.max())
    leaguephase_popularity = league_phase.get(leaguep, league_phase.max())
    stadium_max = stad_max.get(venueId, stad_max.max())

    day_pop1=day_pop.get(day, day_pop.max())
    month_pop1=month_pop.get(month, month_pop.max())
    hour_pop1=hour_pop.get(hour, hour_pop.max())

    X_pred = pd.DataFrame([{
        "home_popularity": home_popularity,
        "away_popularity": away_popularity,
        "city_popularity": city_popularity,
        "country_popularity": country_popularity,
        "league_popularity": league_popularity,
        "stadium_popularity": stadium_popularity,
        "phase_popularity": phase_popularity,
        "leaguephase_popularity":  leaguephase_popularity,
        "stadium_max": stadium_max,
        "month_pop":month_pop1,
        "hour_pop":hour_pop1,
        "day_pop":day_pop1,
        #"team_pop_gap": team_pop_gap,
        #"stadium_league_pop":stadium_league_pop,
        #"leagueName": league,
        #"city": city,
        #"country": country,
        #"is_national": is_national,
        #"home_local": home_local,
        #"away_local": away_local,
        #"neutral_venue": neutral_venue
    }])
    #stadmax=(sinna.groupby("venueId")["attendance"].max())
    #pred = int(pipe.predict(X_pred)[0])
    #if pred > stadmax[venueId]:
     #   return stadmax[venueId]
    #else:
    return int(pipe.predict(X_pred)[0])

In [ ]:
def torneo(df, sucedio,mapss):
    for i in range(len(df)):
        x=df.iloc[i]["homeTeamId"]
        y=df.iloc[i]["awayTeamId"]
        cont = df.iloc[i]["country"]
        cid= df.iloc[i]["city"]
        nat = df.iloc[i]["is_national"]
        lea = df.iloc[i]["leagueName"]
        stad = df.iloc[i]["venueId"]
        fase = df.iloc[i]["phase"]
        
        dia = df.iloc[i]["day_week"]
        mes = df.iloc[i]["month"]
        hora = df.iloc[i]["hour"]
        #cap=df.iloc[i]["stadium_max"]
        cap=stad_max[stad]
        z1 = predecir_asistencia(x, y,cont,cid,lea,stad,nat,fase,dia,mes,hora)
        z2 = predecir_asistencia(y, x,cont,cid,lea,stad,nat,fase,dia,mes,hora)
        z=0
        if sucedio:
            r = df.iloc[i]["attendance"]
            er1 = abs(r-z1)
            er2 = abs(r-z2)
            if er1 < er2:
                z = z1
            else:
                z = z2
            if cap > 74000:
                leaguefase=lea+fase
                z = z+mapss["FIFA World CupFinal"]
                z= int(z)
            elif cap > 50000:
                leaguefase=lea+fase
                z = z+mapss[leaguefase]
                z= int(z)
            print(f'Model: {z}, Real: {r}. {id_name[x]} vs {id_name[y]} in {cid}, {cont}')
        else:
            z = max(z1,z2)
            if cap > 74000:
                leaguefase=lea+fase
                z = z+mapss["FIFA World CupFinal"]
                z= int(z)
            elif cap > 50000:
                leaguefase=lea+fase
                z = z+mapss[leaguefase]
                z= int(z)
            if x==203 and y==467:
                z+=4713
            print(f'Model: {z}. {id_name[x]} vs {id_name[y]} in {cid}, {cont}.')
        df.loc[df.index[i], "modpred"] = z

In [ ]:
def checar(df,ent,mapss):
    if ent:
        torneo(df,ent,mapss)
        print(mean_absolute_error(df["attendance"], df["modpred"]))
    else:
        torneo(df,ent,mapss)

In [ ]:
wc2026=resultdf[wc2026]
wc2026=wc2026.sort_values(by="gameId")

In [ ]:
#PARA 2014 DOBLE MAX (ESTADIO Y LEAGUEPHASE), PARA EL RESTO SOLO MAX EN LEAGUE_PHASE#
#PARA MUNDIAL DE CLUBES min leaguePhase, y max en estadio

In [ ]:
stad_max[10143]=45000
stad_max[4643]=65326
stad_max[1672.0]=87000

#stadium_pop[10143]=45000
#stadium_pop[4643]=65326
wc2026.loc[:, "stadium_max"] = wc2026["venueId"].map(stad_max)

In [ ]:
#city_pop= (sinna.groupby("city")["attendance"].mean())
home_pop = combined.groupby("team")["attendance"].max()
#away_pop= (sinna[sinna["away_local"]==0].groupby("awayTeamId")["attendance"].mean())
stadium_pop=(sinna.groupby("venueId")["attendance"].mean())
phase_pop=(sinna.groupby("phase")["attendance"].mean())
#country_pop=(sinna.groupby("country")["attendance"].mean())
league_phase=(sinna.groupby("leaguePhase")["attendance"].max())
league_pop=(sinna.groupby("leagueName")["attendance"].mean())

In [ ]:
#partido inaugural 4713

In [ ]:
#pd.set_option('display.max_rows', None)
pd.reset_option('display.max_rows')

In [ ]:
mapss={'FIFA World CupStage': 4128.5440414507775,
 'FIFA World Cup16': 3509.6666666666665,
 'FIFA World CupQuarterfinals': 3300.4,
 'FIFA World CupPlace': 3702.75,
 'FIFA World CupSemifinals': 5835.25,
 'FIFA World CupFinal': 12476.0,
 'FIFA World CupRound': 5823.375}

In [ ]:
wc2026.to_csv("results_wc2026.csv",index=False)

In [ ]:
checar(wc2026,0,mapss)

In [ ]:
todo = pd.concat([wc2022,wc2018,wc2014,wc2010])
mapss={}
todo["abs_error"]=abs(todo["attendance"]-todo["modpred"])
for i in todo["leaguePhase"].unique():
    avg=todo[todo["leaguePhase"]==i]["abs_error"].mean()
    #print(f"{i}: {avg}")
    mapss[i]=avg
    mask=(todo["leaguePhase"]==i)
    todo.loc[mask,"+mae"]=todo["modpred"]+avg
    todo.loc[mask,"-mae"]=todo["modpred"]-avg
mapss
#todo[todo["country"]!="South Africa"].sort_values(by="abs_error")
#todo[todo["country"]=="Russia"][["homeTeamName","awayTeamName","attendance","modpred","abs_error","+mae","-mae"]].tail(60)
#todo.to_csv("mundiales.csv",index=False)

In [ ]:
#for i in mapss.keys():
#    mask=(wc2026["leaguePhase"]==i)
#    wc2026.loc[mask,"+mae"]=wc2026["modpred"]+mapss[i]
#    wc2026.loc[mask,"-mae"]=wc2026["modpred"]-mapss[i]

In [ ]:
comb = np.concatenate((cwc["homeTeamName"],cwc["awayTeamName"]))
uniq = np.unique(comb)
total =0
mundc={"pais":[],"mod":[],"real":[]}
for i in uniq:
    cad =  cwc[(cwc["homeTeamName"] == i) | (cwc["awayTeamName"] == i)]
    tot = np.average(cad["attendance"])
    totmod = np.average(cad["modpred"])
    mundc["pais"].append(i)
    mundc["mod"].append(totmod)
    mundc["real"].append(tot)
    print(f'{i} llevó {tot} aficionados al mundial de clubes (mod {totmod})')
    total += tot

In [ ]:
comb = np.concatenate((wc2022["homeTeamName"],wc2022["awayTeamName"]))
uniq = np.unique(comb)
total =0
mund2022={"pais":[],"mod":[],"real":[]}
for i in uniq:
    cad =  wc2022[(wc2022["homeTeamName"] == i) | (wc2022["awayTeamName"] == i)]
    tot = np.average(cad["attendance"])
    totmod = np.average(cad["modpred"])
    mund2022["pais"].append(i)
    mund2022["mod"].append(totmod)
    mund2022["real"].append(tot)
    print(f'{i} llevó {tot} aficionados al mundial de 2022 (mod {totmod})')
    total += tot

In [ ]:
comb = np.concatenate((wc2026["homeTeamName"],wc2026["awayTeamName"]))
uniq = np.unique(comb)
total =0
mund2026={"pais":[],"att":[]}
i = 0
for i in uniq:
    cad =  wc2026[(wc2026["homeTeamName"] == i) | (wc2026["awayTeamName"] == i)]
    totmod = np.average(cad["modpred"])
    mund2026["pais"].append(i)
    mund2026["att"].append(totmod)
    print(f'{i} llevará {totmod} aficionados al mundial de 2026 según el modelo')
    total += totmod

In [ ]:
wc2026[["homeTeamName","awayTeamName","stadium","city","country","modpred"]].head(60).style.hide(axis="index")

In [ ]:
mund2026=pd.DataFrame(mund2026)
mund2022=pd.DataFrame(mund2022)
mundc=pd.DataFrame(mundc)

In [ ]:
wc2026=wc2026.sort_values(by="modpred")
wc2026[["homeTeamName","awayTeamName","stadium","city","country","modpred"]].head(60).style.hide(axis="index")